In [1]:
import pandas as pd
from datetime import timedelta
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
!pip install pmdarima==2.0.3



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 20.7 MB/s eta 0:00:00


# Cicalino

In [3]:
!pip install streamlit pyngrok


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 7.0 MB/s eta 0:00:00


In [4]:
pip install lightgbm catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.3 MB/s eta 0:00:00


In [5]:
%%writefile streamlit_app.py

Writing streamlit_app.py


**clean_weather_data and capture functions**

In [15]:
%%writefile streamlit_app.py
import streamlit as st
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.svm import SVC
import lightgbm as lgb
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Load Cicalino 1
capture_c1 = pd.read_excel("/content/drive/MyDrive/Locations/grafico-delle-catture (Cicalino 1).xlsx")
weather_c1 = pd.read_excel("/content/drive/MyDrive/Locations/dati-meteo-storici (Cicalino 1).xlsx",header=[0, 1])

# Load Cicalino 2
capture_c2 = pd.read_excel("/content/drive/MyDrive/Locations/grafico-delle-catture (Cicalino 2).xlsx")
weather_c2 = pd.read_excel("/content/drive/MyDrive/Locations/dati-meteo-storici (Cicalino 2).xlsx",header=[0, 1])


def run_eda(df):
    df = df.copy()
    st.subheader(" Dataset Info")
    buffer = []
    df.info(buf=buffer.append)
    st.text('\n'.join(buffer))

    st.subheader("📋 Full Dataset Preview")
    st.dataframe(df, use_container_width=True, height=400)

    st.subheader(" Summary Statistics")
    st.dataframe(df.describe())

    st.subheader(" Distribution of New Captures")
    plt.figure(figsize=(6, 4))
    sns.histplot(df['new_captures'], bins=20, kde=True)
    plt.xlabel("New Captures")
    plt.ylabel("Frequency")
    st.pyplot(plt.gcf())
    plt.clf()

    st.subheader(" New Captures Over Time")
    plt.figure(figsize=(10, 4))
    sns.lineplot(data=df, x='datetime', y='new_captures')
    st.pyplot(plt.gcf())
    plt.clf()

    st.subheader(" Correlation Matrix")
    numeric_cols = df.select_dtypes(include=['number']).columns
    plt.figure(figsize=(8, 6))
    sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap='coolwarm')
    st.pyplot(plt.gcf())
    plt.clf()

    for col in ['avg_temp', 'avg_humidity', 'temp_range']:
        if col in df.columns:
            st.subheader(f" New Captures vs {col}")
            plt.figure(figsize=(6, 4))
            sns.boxplot(x=pd.qcut(df[col], q=4, duplicates='drop'), y='new_captures', data=df)
            st.pyplot(plt.gcf())
            plt.clf()

def plot_feature_correlation_heatmap(df: pd.DataFrame, target_col='insect_count', figsize=(10, 6)):
    df = df.copy()
    numeric_cols = df.select_dtypes(include='number').columns
    if target_col not in numeric_cols:
        st.error(f" Target column '{target_col}' not found or not numeric.")
        return None
    corr = df[numeric_cols].corr()[[target_col]].sort_values(by=target_col, ascending=False)
    st.subheader(f" Correlation with Target: `{target_col}`")
    plt.figure(figsize=figsize)
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", cbar=True)
    st.pyplot(plt.gcf())
    plt.clf()
    return corr

import pandas as pd

def clean_weather_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Step 1: Remove header row if it contains non-numeric info
    if df.iloc[0].isna().sum() >= 3 and 'low' in str(df.iloc[0].values) and 'high' in str(df.iloc[0].values):
        df = df.iloc[1:]

    # Step 2: Rename columns to consistent English names
    df.columns = ['datetime', 'avg_temp', 'min_temp', 'max_temp', 'avg_humidity']

    # Step 3: Parse datetime
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce', dayfirst=True)
    df.dropna(subset=['datetime'], inplace=True)

    # Step 4: Clean numeric fields (commas to dots → float)
    for col in ['avg_temp', 'min_temp', 'max_temp', 'avg_humidity']:
        df[col] = (
            df[col].astype(str)
            .str.replace(',', '.', regex=False)
            .str.extract(r'([\d\.]+)')[0]
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df.dropna(subset=['avg_temp', 'min_temp', 'max_temp', 'avg_humidity'], inplace=True)

    # Step 5: Extract date for grouping
    df['date'] = df['datetime'].dt.date

    # Step 6: Optional - loop by date (extendable like capture logic)
    processed_rows = []
    for date, group in df.groupby('date'):
        row = {
            'datetime': pd.to_datetime(date),
            'avg_temp': group['avg_temp'].mean(),
            'min_temp': group['min_temp'].mean(),
            'max_temp': group['max_temp'].mean(),
            'avg_humidity': group['avg_humidity'].mean()
        }
        processed_rows.append(row)

    # Step 7: Return as DataFrame
    return pd.DataFrame(processed_rows)




import pandas as pd

def clean_capture(df):
    df.columns = ['datetime', 'insect_count', 'new_captures', 'reviewed', 'event']
    df['datetime'] = pd.to_datetime(df['datetime'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['datetime'])

    df['insect_count'] = pd.to_numeric(df['insect_count'], errors='coerce')
    df['new_captures'] = pd.to_numeric(df['new_captures'], errors='coerce')
    df['event'] = df['event'].fillna("").astype(str).str.strip()
    df['date'] = df['datetime'].dt.date

    processed_rows = []

    for date, group in df.groupby('date'):
        cleaning_times = group[group['event'] == "Cleaning"]['datetime']
        cleaning_flag = not cleaning_times.empty

        if cleaning_flag:
            group = group[group['datetime'] < cleaning_times.min()]

        if not group.empty:
            last_row = group.sort_values('datetime').iloc[-1]
            insect_count = last_row['insect_count']
            new_captures = last_row['new_captures']
        else:
            insect_count = 0
            new_captures = 0

        processed_rows.append({
            "datetime": pd.to_datetime(date),
            "insect_count": int(insect_count) if pd.notna(insect_count) else 0,
            "new_captures": int(new_captures) if pd.notna(new_captures) else 0,
            "cleaning_event": cleaning_flag
        })

    return pd.DataFrame(processed_rows)

Cicalino1_df_weather = clean_weather_data(weather_c1)

Cicalino1_df_capture=clean_capture(capture_c1)


Cicalino2_df_weather = clean_weather_data(weather_c2)

Cicalino2_df_capture=clean_capture(capture_c2)
def merge_weather_with_capture(weather_df: pd.DataFrame, capture_df: pd.DataFrame) -> pd.DataFrame:

    # Ensure datetime columns are datetime64[ns]
    weather_df['datetime'] = pd.to_datetime(weather_df['datetime'])
    capture_df['datetime'] = pd.to_datetime(capture_df['datetime'])

    # Inner join ensures only matching days are used
    merged = pd.merge(capture_df, weather_df, on='datetime', how='inner')

    return merged


combineed_Cicalino1 = merge_weather_with_capture(Cicalino1_df_weather, Cicalino1_df_capture)
combineed_Cicalino2= merge_weather_with_capture(Cicalino2_df_weather, Cicalino2_df_capture)


def run_eda(df):
    import io
    df = df.copy()

    # st.subheader("📌 Dataset Info")
    # buffer = io.StringIO()
    # df.info(buf=buffer)
    # st.text(buffer.getvalue())
    st.subheader("📋 Full Dataset Preview")
    st.dataframe(df, use_container_width=True, height=400)

    st.subheader(" Distribution of New Captures")
    plt.figure(figsize=(6, 4))
    sns.histplot(df['new_captures'], bins=20, kde=True)
    plt.xlabel("New Captures")
    plt.ylabel("Frequency")
    st.pyplot(plt.gcf())
    plt.clf()

    st.subheader(" New Captures Over Time")
    plt.figure(figsize=(10, 4))
    sns.lineplot(data=df, x='datetime', y='new_captures')
    st.pyplot(plt.gcf())
    plt.clf()

    st.subheader(" Correlation Matrix")
    numeric_cols = df.select_dtypes(include=['number']).columns
    plt.figure(figsize=(8, 6))
    sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap='coolwarm')
    st.pyplot(plt.gcf())
    plt.clf()




def plot_feature_correlation_heatmap(df: pd.DataFrame, target_col='insect_count', figsize=(10, 6)):
    df = df.copy()
    numeric_cols = df.select_dtypes(include='number').columns
    if target_col not in numeric_cols:
        st.error(f" Target column '{target_col}' not found or not numeric.")
        return None
    corr = df[numeric_cols].corr()[[target_col]].sort_values(by=target_col, ascending=False)
    st.subheader(f"Correlation with Target: `{target_col}`")
    plt.figure(figsize=figsize)
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", cbar=True)
    st.pyplot(plt.gcf())
    plt.clf()
    return corr

def add_features(df: pd.DataFrame, lags: list = [ 2, 3], rolling_window: int = 3) -> pd.DataFrame:

    df = df.copy()
    df = df.sort_values("datetime").reset_index(drop=True)

    # Lag features
    for lag in lags:
        df[f"lag_{lag}"] = df["insect_count"].shift(lag)

    # Rolling mean of target
    df["rolling_mean"] = df["insect_count"].shift(1).rolling(window=rolling_window).mean()

    # Interaction feature
    df["humidity_temp_interaction"] = df["avg_temp"] * df["avg_humidity"]

    df.dropna(inplace=True)

    return df
def combine_locations(df1: pd.DataFrame, df2: pd.DataFrame, name1='Cicalino 1', name2='Cicalino 2') -> pd.DataFrame:
    """
    Combines two location datasets and adds a 'location' column.
    """
    df1 = df1.copy()
    df2 = df2.copy()

    df1['location'] = name1
    df2['location'] = name2

    combined = pd.concat([df1, df2], ignore_index=True)
    return combined
background_image_url = "https://m.media-amazon.com/images/I/91zRe4HJr0S.jpg"

st.markdown(f"""
    <style>
    /* Sidebar background image with soft dark overlay */
    [data-testid="stSidebar"] {{
        background: linear-gradient(rgba(0,0,0,0.3), rgba(0,0,0,0.3)), url("{background_image_url}");
        background-size: cover;
        background-position: center;
        background-repeat: no-repeat;
    }}

     /* Sidebar Title */
    [data-testid="stSidebar"] h1, [data-testid="stSidebar"] h2 {{
        color: white !important;     /* <-- Sidebar Title in White */
        font-size: 26px !important;
        text-align: center !important;
        font-weight: bold !important;
        margin-bottom: 30px !important;
    }}


    /* Sidebar Buttons */
    .element-container button {{
        width: 100% !important;
        min-height: 50px !important;
        font-size: 18px !important;
        font-weight: bold !important;
        color: black !important;
        background-color: white !important;
        border: none;
        border-radius: 10px;
        margin-top: 10px;
        transition: all 0.3s ease-in-out;
    }}

    /* Button text inside */
    [data-testid="baseButton-secondary"] span {{
        display: block;
        width: 100%;
        text-align: center;
    }}

    /* Button hover */
    .element-container button:hover {{
        background-color: rgba(255, 255, 255, 0.9) !important;
        transform: scale(1.03);
        box-shadow: 0 4px 12px rgba(0,0,0,0.4);
        color: black !important;
    }}

    /* Sidebar scroll if overflow */
    .sidebar-content {{
        overflow-y: auto;
        height: 100%;
    }}



    </style>

""", unsafe_allow_html=True)






# Sidebar title
st.sidebar.markdown("<h2>Insect Capture Dashboard</h2>", unsafe_allow_html=True)

# 1. Define sidebar menu with Weather option
if "page" not in st.session_state:
    st.session_state.page = "Weather Overview"  # default page is Weather

menu_items = {
    "Weather Overview": "",

    "Cicalino": "",
    "Imola": "",
    "Final Dataset": "",
    "Regression Models": "",
    "Classification Models": ""
}

for page, icon in menu_items.items():
    if st.sidebar.button(f"{icon} {page}", key=f"btn_{page}"):
        st.session_state.page = page

current_page = st.session_state.page

# 2. If Weather Overview Page
if current_page == "Weather Overview":
    st.header("🌤️ Weather Overview")

    location_choice = st.selectbox("Select Dataset", ["Cicalino", "Imola", "Final Dataset"])
    # --- Create final_cicalino ---
    combineed_Cicalino1_features = add_features(combineed_Cicalino1)
    combineed_Cicalino2_features = add_features(combineed_Cicalino2)

    combineed_Cicalino1_features['location'] = 'Cicalino 1'
    combineed_Cicalino2_features['location'] = 'Cicalino 2'

    final_cicalino = pd.concat([combineed_Cicalino1_features, combineed_Cicalino2_features], ignore_index=True)
    final_cicalino['location'] = 'Cicalino'

    # --- Create final_imola ---
    capture_i1 = pd.read_excel("/content/drive/MyDrive/Locations/grafico-delle-catture (Imola 1).xlsx")
    weather_i1 = pd.read_excel("/content/drive/MyDrive/Locations/dati-meteo-storici (Imola 1).xlsx", header=[0, 1])
    capture_i2 = pd.read_excel("/content/drive/MyDrive/Locations/grafico-delle-catture (Imola 2).xlsx")
    weather_i2 = pd.read_excel("/content/drive/MyDrive/Locations/dati-meteo-storici (Imola 2).xlsx", header=[0, 1])
    capture_i3 = pd.read_excel("/content/drive/MyDrive/Locations/grafico-delle-catture (Imola 3).xlsx")
    weather_i3 = pd.read_excel("/content/drive/MyDrive/Locations/dati-meteo-storici (Imola 3).xlsx", header=[0, 1])

    df_i1 = merge_weather_with_capture(clean_weather_data(weather_i1), clean_capture(capture_i1))
    df_i2 = merge_weather_with_capture(clean_weather_data(weather_i2), clean_capture(capture_i2))
    df_i3 = merge_weather_with_capture(clean_weather_data(weather_i3), clean_capture(capture_i3))

    final_imola = pd.concat([df_i1, df_i2, df_i3], ignore_index=True)
    final_imola = add_features(final_imola)
    final_imola['location'] = 'Imola'

    # --- Create final_dataset ---
    final_dataset = pd.concat([final_cicalino, final_imola], ignore_index=True)
    final_dataset = final_dataset.sort_values('datetime').reset_index(drop=True)

    if location_choice == "Cicalino":
        summary_df = final_cicalino.copy()
    elif location_choice == "Imola":
        summary_df = final_imola.copy()
    else:
        summary_df = final_dataset.copy()

     # Basic weather metrics
    col1, col2, col3 = st.columns(3)
    col1.metric("Average Temp (°C)", f"{summary_df['avg_temp'].mean():.2f}")
    col2.metric("Average Humidity (%)", f"{summary_df['avg_humidity'].mean():.2f}")
    col3.metric("Total Days", f"{summary_df['datetime'].nunique()}")
    # Line plot for temperature
    st.subheader(" Average Temperature Over Time")
    fig_temp, ax_temp = plt.subplots(figsize=(10, 4))
    ax_temp.plot(summary_df['datetime'], summary_df['avg_temp'], color='orange')
    ax_temp.set_ylabel("Avg Temperature (°C)")
    ax_temp.set_xlabel("Date")
    st.pyplot(fig_temp)

    # Line plot for humidity
    st.subheader(" Average Humidity Over Time")
    fig_humidity, ax_humidity = plt.subplots(figsize=(10, 4))
    ax_humidity.plot(summary_df['datetime'], summary_df['avg_humidity'], color='blue')
    ax_humidity.set_ylabel("Avg Humidity (%)")
    ax_humidity.set_xlabel("Date")
    st.pyplot(fig_humidity)

        # --- Temperature Trends Plot ---
    st.subheader(" Temperature Trends")
    fig_temp_trends, ax_temp_trends = plt.subplots(figsize=(10, 4))
    ax_temp_trends.plot(summary_df['datetime'], summary_df['avg_temp'], label="Average Temp", marker='o')
    ax_temp_trends.plot(summary_df['datetime'], summary_df['max_temp'], label="Max Temp", marker='o')
    ax_temp_trends.plot(summary_df['datetime'], summary_df['min_temp'], label="Min Temp", marker='o')
    ax_temp_trends.set_xlabel("Date")
    ax_temp_trends.set_ylabel("Temperature")
    ax_temp_trends.set_title("Temperature Trends")
    ax_temp_trends.legend()
    st.pyplot(fig_temp_trends)







if current_page == "Cicalino":
                st.header("📍 Cicalino Data")
                st.subheader("EDA")
                with st.expander(" Cicalino 1"):
                  run_eda(combineed_Cicalino1)
                  combineed_Cicalino_feature = add_features(combineed_Cicalino1)
                  correlations = plot_feature_correlation_heatmap(combineed_Cicalino_feature, target_col="insect_count")

                if correlations is not None:
                    pass  # or just delete this block entirely


                with st.expander(" Cicalino 2"):
                    run_eda(combineed_Cicalino2)
                    combineed_Cicalino2_features = add_features(combineed_Cicalino2)
                    correlations = plot_feature_correlation_heatmap(combineed_Cicalino2_features, target_col="insect_count")

                if correlations is not None:
                        pass
                with st.expander(" Merged Cicalino 1 and Cicalino 2"):

                    combineed_Cicalino_feature['location'] = 'Cicalino 1'
                    combineed_Cicalino2_features['location'] = 'Cicalino 2'

                    combined_Cicalino_feature = pd.concat([combineed_Cicalino_feature, combineed_Cicalino2_features], ignore_index=True)


                    # Assume: merged_c1 and merged_c2 are pre-cleaned DataFrames for both locations
                    final_cicalino = combine_locations(combineed_Cicalino_feature, combineed_Cicalino2_features, name1='Cicalino 1', name2='Cicalino 2')

                    st.dataframe(final_cicalino)


                st.subheader("Regression Models on Final Cicalino Dataset")
                import statsmodels.api as sm
                from statsmodels.discrete.count_model import ZeroInflatedPoisson
                from statsmodels.discrete.discrete_model import NegativeBinomial
                from sklearn.linear_model import LinearRegression, PoissonRegressor
                from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
                from sklearn.model_selection import train_test_split
                from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



                # Features and target
                features = [
                    'avg_temp', 'avg_humidity',
                    'lag_2',
                    'rolling_mean'
                ]
                X = final_cicalino[features]
                y = final_cicalino['insect_count']

                # Split train/test
                X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

                # Helper function to evaluate
                def evaluate_model(name, y_true, y_pred):
                    st.markdown(f"#### {name}")
                    st.write(f"**MAE**: {mean_absolute_error(y_true, y_pred):.2f}")
                    st.write(f"**MSE**: {mean_squared_error(y_true, y_pred):.2f}")
                    st.write(f"**R² Score**: {r2_score(y_true, y_pred):.2f}")
                    st.line_chart(pd.DataFrame({'Actual': y_true.values, 'Predicted': y_pred}).reset_index(drop=True))

                with st.expander("Linear Regression"):
                    linreg = LinearRegression()
                    linreg.fit(X_train, y_train)
                    y_pred_linreg = linreg.predict(X_test)
                    evaluate_model("Linear Regression", y_test, y_pred_linreg)


                with st.expander(" Random Forest Regression"):
                    rf = RandomForestRegressor(n_estimators=100, random_state=42)
                    rf.fit(X_train, y_train)
                    y_pred_rf = rf.predict(X_test)
                    evaluate_model("Random Forest Regression", y_test, y_pred_rf)

                with st.expander(" Gradient Boosting Regression"):
                    gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
                    gb.fit(X_train, y_train)
                    y_pred_gb = gb.predict(X_test)
                    evaluate_model("Gradient Boosting Regression", y_test, y_pred_gb)




                with st.expander("Zero-Inflated Poisson Regression"):
                    X_train_const = sm.add_constant(X_train)
                    X_test_const = sm.add_constant(X_test)

                    zip_model = ZeroInflatedPoisson(y_train, X_train_const).fit()
                    y_pred_zip = zip_model.predict(X_test_const)
                    evaluate_model("Zero-Inflated Poisson", y_test, y_pred_zip)

                with st.expander("Negative Binomial Regression"):
                    nb_model = NegativeBinomial(y_train, X_train_const).fit()
                    y_pred_nb = nb_model.predict(X_test_const)
                    evaluate_model("Negative Binomial", y_test, y_pred_nb)


                import statsmodels.api as sm
                from statsmodels.tsa.stattools import adfuller
                import numpy as np
                import matplotlib.pyplot as plt
                import pandas as pd
                import matplotlib.pyplot as plt
                import numpy as np
                import statsmodels.api as sm
                from statsmodels.tsa.stattools import adfuller
                from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
                from sklearn.metrics import mean_squared_error


                with st.expander("Time Series Modeling"):


                    import itertools
                    from statsmodels.tsa.statespace.sarimax import SARIMAX
                    from sklearn.metrics import mean_squared_error, r2_score
                    import numpy as np
                    import matplotlib.pyplot as plt
                    import statsmodels.api as sm
                    from statsmodels.tsa.stattools import adfuller

                    # Step 1: Aggregate daily data
                    daily_cicalino = final_cicalino.groupby('datetime').agg({
                        'insect_count': 'sum',
                        'avg_temp': 'mean',
                        'avg_humidity': 'mean'
                    }).sort_index()

                    st.write("Start Date:", daily_cicalino.index.min())
                    st.write("End Date:", daily_cicalino.index.max())

                    # Step 2: Prepare target and features
                    y = daily_cicalino['insect_count']
                    exog = daily_cicalino[['avg_temp', 'avg_humidity']]

                    fig1, ax1 = plt.subplots(figsize=(10, 4))
                    ax1.plot(daily_cicalino['insect_count'])
                    ax1.set_title("Original Daily Insect Counts")
                    st.pyplot(fig1)

                    # Step 3: Stationarity Check
                    d = 0
                    adf_result = adfuller(y)
                    st.info(f"Original ADF p-value: {adf_result[1]:.5f}")

                    if adf_result[1] > 0.05:
                        d = 1
                        st.warning("Applying First Differencing (d=1)")
                        y = y.diff().dropna()

                        fig2, ax2 = plt.subplots(figsize=(10, 4))
                        ax2.plot(y)
                        ax2.set_title("After First Differencing")
                        st.pyplot(fig2)

                        adf_result = adfuller(y)
                        st.info(f"After First Differencing ADF p-value: {adf_result[1]:.5f}")

                        if adf_result[1] > 0.05:
                            d = 2
                            st.warning("Still not stationary, applying Second Differencing (d=2)")
                            y = y.diff().dropna()

                            fig3, ax3 = plt.subplots(figsize=(10, 4))
                            ax3.plot(y)
                            ax3.set_title("After Second Differencing")
                            st.pyplot(fig3)

                            adf_result = adfuller(y)
                            st.info(f"After Second Differencing ADF p-value: {adf_result[1]:.5f}")

                    st.success(f"Final Differencing Order Used: d={d}")

                    # Step 4: ACF and PACF after differencing
                    st.subheader("ACF and PACF After Differencing")
                    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
                    plot_acf(y, lags=20, ax=axes[0])
                    plot_pacf(y, lags=20, ax=axes[1])
                    st.pyplot(fig)

                    # Step 5: SARIMAX Grid Search
                    st.subheader("Hyperparameter Tuning (Grid Search)")

                    def sarimax_grid_search(y, exog, pdq, seasonal_pdq):
                        best_score = float("inf")
                        best_cfg = None
                        best_seasonal = None
                        best_model = None

                        for order in pdq:
                            for seasonal_order in seasonal_pdq:
                                try:
                                    model = SARIMAX(y,
                                                    exog=exog,
                                                    order=order,
                                                    seasonal_order=seasonal_order,
                                                    enforce_stationarity=False,
                                                    enforce_invertibility=False)
                                    results = model.fit(disp=False)

                                    pred = results.predict(start=y.index[int(len(y)*0.8)],
                                                          end=y.index[-1],
                                                          exog=exog.iloc[int(len(y)*0.8):])

                                    rmse = np.sqrt(mean_squared_error(y.iloc[int(len(y)*0.8):], pred))

                                    if rmse < best_score:
                                        best_score = rmse
                                        best_cfg = order
                                        best_seasonal = seasonal_order
                                        best_model = results
                                except Exception as e:
                                    continue

                        return best_cfg, best_seasonal, best_model

                    pdq = list(itertools.product(range(0, 3), range(0, 3), range(0, 3)))  # (p,d,q)
                    seasonal_pdq = [(x, 0, y, 7) for x in range(0, 2) for y in range(0, 2)]  # (P,D,Q,s)

                    def sarimax_grid_search(y, exog, pdq, seasonal_pdq):
                        best_score = float("inf")
                        best_cfg = None
                        best_seasonal = None
                        best_model = None

                        for order in pdq:
                            for seasonal_order in seasonal_pdq:
                                try:
                                    model = SARIMAX(y,
                                                    exog=exog,
                                                    order=order,
                                                    seasonal_order=seasonal_order,
                                                    enforce_stationarity=False,
                                                    enforce_invertibility=False)
                                    results = model.fit(disp=False)

                                    pred = results.predict(start=y.index[int(len(y)*0.8)],
                                                          end=y.index[-1],
                                                          exog=exog.iloc[int(len(y)*0.8):])

                                    rmse = np.sqrt(mean_squared_error(y.iloc[int(len(y)*0.8):], pred))

                                    if rmse < best_score:
                                        best_score = rmse
                                        best_cfg = order
                                        best_seasonal = seasonal_order
                                        best_model = results
                                except Exception as e:
                                    continue

                        return best_cfg, best_seasonal, best_model








                    best_order, best_seasonal_order, best_model = sarimax_grid_search(daily_cicalino['insect_count'], exog, pdq, seasonal_pdq)

                    st.success(f"Best SARIMAX Order: {best_order}")
                    st.success(f"Best Seasonal Order: {best_seasonal_order}")

                    # Step 6: Forecast
                    st.subheader("Forecast Next 30 Days")

                    last_known_exog = exog.iloc[-1]
                    future_exog = pd.DataFrame(
                        np.tile(last_known_exog.values, (30, 1)),
                        columns=exog.columns
                    )

                    forecast = best_model.get_forecast(steps=30, exog=future_exog)
                    forecast_index = pd.date_range(daily_cicalino.index[-1]+pd.Timedelta(days=1), periods=30, freq='D')
                    forecast_ci = forecast.conf_int()

                    fig, ax = plt.subplots(figsize=(12, 6))
                    ax.plot(daily_cicalino['insect_count'], label="Observed")
                    ax.plot(forecast_index, forecast.predicted_mean, label="Forecast", color='red')
                    ax.fill_between(forecast_index, forecast_ci.iloc[:,0], forecast_ci.iloc[:,1], color='pink', alpha=0.3)
                    ax.legend()
                    st.pyplot(fig)

                    # Step 7: Evaluate on Test Set
                    st.subheader(" Model Evaluation on Test Set")

                    split_idx = int(len(y) * 0.8)
                    y_test = y.iloc[split_idx:]
                    exog_test = exog.iloc[split_idx:]

                    pred_test = best_model.predict(start=y_test.index[0], end=y_test.index[-1], exog=exog_test)
                    rmse = np.sqrt(mean_squared_error(y_test, pred_test))
                    r2 = r2_score(y_test, pred_test)

                    st.info(f" RMSE: {rmse:.3f}")
                    st.info(f" R² Score: {r2:.3f}")

                    fig, ax = plt.subplots(figsize=(10, 4))
                    ax.plot(y_test.index, y_test, label="Actual")
                    ax.plot(y_test.index, pred_test, label=f"SARIMAX {best_order} Seasonal {best_seasonal_order}", linestyle='--')
                    ax.set_title("Final SARIMAX Forecast on Test Set")
                    ax.legend()
                    ax.grid(True)
                    st.pyplot(fig)

                with st.expander(" Best Model of Regression "):
                    st.markdown("---")
                    best_regression_metrics = {
                        "Model": "Gradient Boosting Regression",
                        "MAE": 0.23,
                        "MSE": 0.14,
                        "R² Score": 0.66
                    }

                    st.success(
                        f" The best regression model is **{best_regression_metrics['Model']}** with:\n"
                        f"- **MAE**: {best_regression_metrics['MAE']:.2f}  \n"
                        f"- **MSE**: {best_regression_metrics['MSE']:.2f}  \n"
                        f"- **R² Score**: {best_regression_metrics['R² Score']:.2f}"
                    )


                from sklearn.model_selection import train_test_split
                from sklearn.linear_model import LogisticRegression
                from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
                from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
                from sklearn.preprocessing import StandardScaler
                from imblearn.over_sampling import SMOTE
                from sklearn.model_selection import train_test_split
                from sklearn.linear_model import LogisticRegression
                from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
                from sklearn.neighbors import KNeighborsClassifier
                from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
                import matplotlib.pyplot as plt
                import seaborn as sns
                from sklearn.pipeline import Pipeline
                from xgboost import XGBClassifier
                from sklearn.svm import SVC


                # Prepare classification target: binary 'new_captures' (1 if new_captures > 0)
                final_cicalino['label'] = (final_cicalino['new_captures'] > 0).astype(int)


                features = [
                    'avg_temp', 'avg_humidity',
                    'lag_2', 'rolling_mean'
                ]

                final_cicalino['label'] = (final_cicalino['new_captures'] > 0).astype(int)
                X_cls = final_cicalino[features]
                y_cls = final_cicalino['label']


                X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
                    X_cls, y_cls, test_size=0.3, stratify=y_cls, random_state=42
                )

                # Apply SMOTE to balance the training set
                sm = SMOTE(random_state=42)
                X_train_cls_bal, y_train_cls_bal = sm.fit_resample(X_train_cls, y_train_cls)

                # def evaluate_classifier(name, y_true, y_pred):
                #     st.markdown(f"#### {name}")
                #     st.write(f"**Accuracy**: {accuracy_score(y_true, y_pred):.2f}")
                #     st.write(f"**Precision**: {precision_score(y_true, y_pred):.2f}")
                #     st.write(f"**Recall**: {recall_score(y_true, y_pred):.2f}")
                #     st.write(f"**F1 Score**: {f1_score(y_true, y_pred):.2f}")
                #     cm = confusion_matrix(y_true, y_pred)
                #     fig, ax = plt.subplots()
                #     sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
                #     ax.set_xlabel("Predicted")
                #     ax.set_ylabel("Actual")
                #     st.pyplot(fig)
                def evaluate_classifier(name, y_true, y_pred):
                    st.markdown(f"#### {name}")
                    st.write(f"**Accuracy**: {accuracy_score(y_true, y_pred):.2f}")
                    st.write(f"**Precision**: {precision_score(y_true, y_pred):.2f}")
                    st.write(f"**Recall**: {recall_score(y_true, y_pred):.2f}")
                    st.write(f"**F1 Score**: {f1_score(y_true, y_pred):.2f}")

                    cm = confusion_matrix(y_true, y_pred)

                    # Check that confusion matrix is valid
                    if cm is not None and cm.size > 0:
                        fig, ax = plt.subplots()
                        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
                        ax.set_xlabel("Predicted")
                        ax.set_ylabel("Actual")
                        st.pyplot(fig)
                        plt.clf()  # <-- clear figure after plot
                    else:
                        st.warning("⚠️ Confusion matrix is empty. Cannot plot.")

                st.subheader("Classification Models on Final Cicalino Dataset")

                with st.expander(" Logistic Regression"):
                    logistic_pipeline = Pipeline([
                        ('scaler', StandardScaler()),
                        ('clf', LogisticRegression(max_iter=1000, random_state=42))
                    ])
                    logistic_pipeline.fit(X_train_cls_bal, y_train_cls_bal)
                    y_pred_lr = logistic_pipeline.predict(X_test_cls)
                    evaluate_classifier("Logistic Regression", y_test_cls, y_pred_lr)

                with st.expander(" Random Forest Classifier"):
                    clf_rf = RandomForestClassifier(n_estimators=100, random_state=42)
                    clf_rf.fit(X_train_cls_bal, y_train_cls_bal)
                    y_pred_rf = clf_rf.predict(X_test_cls)
                    evaluate_classifier("Random Forest", y_test_cls, y_pred_rf)

                with st.expander(" Gradient Boosting Classifier"):
                    clf_gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
                    clf_gb.fit(X_train_cls_bal, y_train_cls_bal)
                    y_pred_gb = clf_gb.predict(X_test_cls)
                    evaluate_classifier("Gradient Boosting", y_test_cls, y_pred_gb)

                with st.expander(" K-Nearest Neighbors Classifier"):
                    knn_pipeline = Pipeline([
                        ('scaler', StandardScaler()),
                        ('clf', KNeighborsClassifier(n_neighbors=5))
                    ])
                    knn_pipeline.fit(X_train_cls_bal, y_train_cls_bal)
                    y_pred_knn = knn_pipeline.predict(X_test_cls)
                    evaluate_classifier("K-Nearest Neighbors (k=5)", y_test_cls, y_pred_knn)



                with st.expander(" XGBoost Classifier"):
                    clf_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
                    clf_xgb.fit(X_train_cls_bal, y_train_cls_bal)
                    y_pred_xgb = clf_xgb.predict(X_test_cls)
                    evaluate_classifier("XGBoost", y_test_cls, y_pred_xgb)




                with st.expander(" Best Model OF Classification"):

                    st.markdown("---")
                    best_model_metrics = {
                        "Accuracy": 0.78,
                        "Precision": 0.33,
                        "Recall": 0.50,
                        "F1 Score": 0.40
                    }

                    st.success(
                        f" The best model is **Random Forest** with:\n"
                        f"- **Accuracy**: {best_model_metrics['Accuracy']:.2f}  \n"
                        f"- **Precision**: {best_model_metrics['Precision']:.2f}  \n"
                        f"- **Recall**: {best_model_metrics['Recall']:.2f}  \n"
                        f"- **F1 Score**: {best_model_metrics['F1 Score']:.2f}"
                    )




# 📍 Imola Data

if current_page == "Imola":
    st.header(" 📍Imola Data")

    import itertools
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.linear_model import LinearRegression
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from imblearn.over_sampling import SMOTE
    from xgboost import XGBClassifier
    import statsmodels.api as sm
    from statsmodels.tsa.stattools import adfuller
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

    # 🛠️ Helper functions
    def evaluate_model(name, y_true, y_pred):
        st.markdown(f"#### {name}")
        st.write(f"**MAE**: {mean_absolute_error(y_true, y_pred):.2f}")
        st.write(f"**MSE**: {mean_squared_error(y_true, y_pred):.2f}")
        st.write(f"**R² Score**: {r2_score(y_true, y_pred):.2f}")
        st.line_chart(pd.DataFrame({'Actual': y_true.values, 'Predicted': y_pred}).reset_index(drop=True))

    def evaluate_classifier(name, y_true, y_pred):
        st.markdown(f"#### {name}")
        st.write(f"**Accuracy**: {accuracy_score(y_true, y_pred):.2f}")
        st.write(f"**Precision**: {precision_score(y_true, y_pred):.2f}")
        st.write(f"**Recall**: {recall_score(y_true, y_pred):.2f}")
        st.write(f"**F1 Score**: {f1_score(y_true, y_pred):.2f}")
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots()
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        st.pyplot(fig)

    # Load Imola datasets
    capture_i1 = pd.read_excel("/content/drive/MyDrive/Locations/grafico-delle-catture (Imola 1).xlsx")
    weather_i1 = pd.read_excel("/content/drive/MyDrive/Locations/dati-meteo-storici (Imola 1).xlsx", header=[0, 1])

    capture_i2 = pd.read_excel("/content/drive/MyDrive/Locations/grafico-delle-catture (Imola 2).xlsx")
    weather_i2 = pd.read_excel("/content/drive/MyDrive/Locations/dati-meteo-storici (Imola 2).xlsx", header=[0, 1])

    capture_i3 = pd.read_excel("/content/drive/MyDrive/Locations/grafico-delle-catture (Imola 3).xlsx")
    weather_i3 = pd.read_excel("/content/drive/MyDrive/Locations/dati-meteo-storici (Imola 3).xlsx", header=[0, 1])

    # Merge and clean
    df_i1 = merge_weather_with_capture(clean_weather_data(weather_i1), clean_capture(capture_i1))
    df_i2 = merge_weather_with_capture(clean_weather_data(weather_i2), clean_capture(capture_i2))
    df_i3 = merge_weather_with_capture(clean_weather_data(weather_i3), clean_capture(capture_i3))

    final_imola = pd.concat([df_i1, df_i2, df_i3], ignore_index=True)
    final_imola = add_features(final_imola)

    # --- EDA ---
    st.subheader("EDA on Merged Imola Dataset")
    with st.expander("EDA Results"):
        run_eda(final_imola)
        plot_feature_correlation_heatmap(final_imola, target_col="insect_count")

    # --- Regression Models ---
    st.subheader("Regression Models on Final Imola Dataset")



        # Prepare classification target
    final_imola['label'] = (final_imola['new_captures'] > 0).astype(int)

    # Features
    features = ['avg_temp', 'avg_humidity','lag_3','rolling_mean']
    X_cls_imola = final_imola[features]
    y_cls_imola = final_imola['label']

    # Split data
    X_train_cls_imola, X_test_cls_imola, y_train_cls_imola, y_test_cls_imola = train_test_split(
        X_cls_imola, y_cls_imola, test_size=0.3, stratify=y_cls_imola, random_state=42
    )

    # Apply SMOTE
    smote = SMOTE(random_state=42)

    X_train_cls_bal, y_train_cls_bal = smote.fit_resample(X_train_cls_imola, y_train_cls_imola)

    with st.expander("Linear Regression"):
        linreg = LinearRegression()
        linreg.fit(X_train_cls_bal, y_train_cls_bal)
        y_pred_linreg = linreg.predict(X_test_cls_imola)
        evaluate_model("Linear Regression", y_test_cls_imola, y_pred_linreg)

    with st.expander("Random Forest Regression"):
        rf = RandomForestRegressor(n_estimators=100, random_state=42)
        rf.fit(X_train_cls_bal, y_train_cls_bal)
        y_pred_rf = rf.predict(X_test_cls_imola)
        evaluate_model("Random Forest Regression", y_test_cls_imola, y_pred_rf)

    with st.expander("Gradient Boosting Regression"):
        gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
        gb.fit(X_train_cls_bal, y_train_cls_bal)
        y_pred_gb = gb.predict(X_test_cls_imola)
        evaluate_model("Gradient Boosting Regression", y_test_cls_imola, y_pred_gb)
    with st.expander(" Time Series Modeling for Imola Dataset"):

      import itertools
      import numpy as np
      import pandas as pd
      import matplotlib.pyplot as plt
      import statsmodels.api as sm
      from statsmodels.tsa.stattools import adfuller
      from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
      from sklearn.metrics import mean_squared_error, r2_score
      from statsmodels.tsa.statespace.sarimax import SARIMAX


      # Prepare data
      df_i1['location'] = 'imola 1'
      df_i2['location'] = 'imola 2'
      combined_imola = pd.concat([df_i1, df_i2], ignore_index=True)

      daily_imola = combined_imola.groupby('datetime').agg({
          'insect_count': 'sum',
          'avg_temp': 'mean',
          'avg_humidity': 'mean',

      }).sort_index()

      # daily_imola['humidity_temp_interaction'] = daily_imola['avg_temp'] * daily_imola['avg_humidity']
      # daily_imola['lag_1'] = daily_imola['insect_count'].shift(1)
      # daily_imola['rolling_mean'] = daily_imola['insect_count'].shift(1).rolling(window=3).mean()
      # daily_imola.dropna(inplace=True)

      y_imola = daily_imola['insect_count']
      X_imola = daily_imola[['avg_temp', 'avg_humidity']]

      st.write("Total points:", len(y_imola))

      # Plot original
      st.subheader("Original Daily Insect Count (Imola)")
      fig, ax = plt.subplots(figsize=(10, 4))
      ax.plot(y_imola, marker='o')
      st.pyplot(fig)

      # Stationarity Check
      st.subheader("Stationarity Testing (ADF Test)")

      adf_test = adfuller(y_imola)
      st.info(f"ADF p-value on original: {adf_test[1]:.5f}")

      d = 0
      if adf_test[1] > 0.05:
          d = 1
          st.warning("Applying First Differencing (d=1)")
          y_imola = y_imola.diff().dropna()

          fig, ax = plt.subplots(figsize=(10, 4))
          ax.plot(y_imola, marker='o')
          ax.set_title("After First Differencing")
          st.pyplot(fig)

          adf_test_1 = adfuller(y_imola)
          st.info(f"ADF p-value after 1st differencing: {adf_test_1[1]:.5f}")

          if adf_test_1[1] > 0.05:
              d = 2
              st.warning("Still non-stationary. Applying Second Differencing (d=2)")
              y_imola = y_imola.diff().dropna()

              fig, ax = plt.subplots(figsize=(10, 4))
              ax.plot(y_imola, marker='o')
              ax.set_title("After Second Differencing")
              st.pyplot(fig)

              adf_test_2 = adfuller(y_imola)
              st.info(f"ADF p-value after 2nd differencing: {adf_test_2[1]:.5f}")

      st.success(f" Final Differencing Order Used: d={d}")

      # ACF and PACF
      st.subheader("ACF and PACF After Differencing")
      max_lag = max(5, min(10, len(y_imola)//2))
      fig, axes = plt.subplots(1, 2, figsize=(14, 4))
      plot_acf(y_imola, lags=max_lag, ax=axes[0])
      plot_pacf(y_imola, lags=max_lag, ax=axes[1])
      st.pyplot(fig)

      # SARIMAX Grid Search
      st.subheader("SARIMAX Hyperparameter Tuning")

      def sarimax_grid_search(y, exog, pdq):
          best_score = float("inf")
          best_cfg = None
          best_model = None

          for order in pdq:
              try:
                  model = SARIMAX(
                      y,
                      exog=exog.iloc[d:],
                      order=order,
                      enforce_stationarity=False,
                      enforce_invertibility=False
                  )
                  results = model.fit(disp=False)

                  pred = results.predict(
                      start=y.index[int(len(y)*0.8)],
                      end=y.index[-1],
                      exog=exog.iloc[int(len(y)*0.8) + d:]
                  )

                  rmse = np.sqrt(mean_squared_error(y.iloc[int(len(y)*0.8):], pred))

                  if rmse < best_score:
                      best_score = rmse
                      best_cfg = order
                      best_model = results
              except Exception as e:
                  continue

          return best_cfg, best_model

      pdq = list(itertools.product(range(0, 3), [d], range(0, 3)))

      best_order_imola, best_model_imola = sarimax_grid_search(y_imola, X_imola, pdq)

      if best_model_imola is not None:
          st.success(f" Best SARIMAX Order Found: {best_order_imola}")

          # Forecast on Test Set
          split_idx = int(len(y_imola) * 0.8)
          y_test = y_imola.iloc[split_idx:]
          X_test = X_imola.iloc[split_idx+d:]

          pred = best_model_imola.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_test)

          # Plot Forecast
          fig, ax = plt.subplots(figsize=(12, 6))
          ax.plot(y_imola, label="Observed", marker='o')
          ax.plot(y_test.index, pred, label="Forecast", color='red', marker='x')
          ax.legend()
          ax.set_title("Observed vs Forecasted (Imola SARIMAX)")
          st.pyplot(fig)

          # Evaluation
          st.subheader("Model Evaluation")
          rmse = np.sqrt(mean_squared_error(y_test, pred))
          r2 = r2_score(y_test, pred)

          st.info(f"**RMSE:** {rmse:.3f}")
          st.info(f"**R² Score:** {r2:.3f}")

      else:
          st.error("No valid SARIMAX model found.")



    # --- Classification Models ---
    st.subheader("Classification Models on Final Imola Dataset")

    with st.expander("Random Forest Classifier"):
        rf_cls = RandomForestClassifier(random_state=42)
        rf_cls.fit(X_train_cls_bal, y_train_cls_bal)
        y_pred_rf = rf_cls.predict(X_test_cls_imola)
        evaluate_classifier("Random Forest", y_test_cls_imola, y_pred_rf)


    with st.expander("XGBoost Classifier"):
        xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
        xgb.fit(X_train_cls_bal, y_train_cls_bal)
        y_pred_xgb = xgb.predict(X_test_cls_imola)
        evaluate_classifier("XGBoost", y_test_cls_imola, y_pred_xgb)

    with st.expander("Decision Tree Classifier"):
        dt = DecisionTreeClassifier(random_state=42)
        dt.fit(X_train_cls_bal, y_train_cls_bal)
        y_pred_dt = dt.predict(X_test_cls_imola)
        evaluate_classifier("Decision Tree", y_test_cls_imola, y_pred_dt)


    with st.expander(" Support Vector Machine (SVC)"):
        svc = SVC(random_state=42)
        svc.fit(X_train_cls_bal, y_train_cls_bal)
        y_pred_svc = svc.predict(X_test_cls_imola)
        evaluate_classifier("Support Vector Machine", y_test_cls_imola, y_pred_svc)
        # with st.expander("Best Classification Model"):
        #     st.success("The best classifier model is **Random Forest** based on overall metrics.")



# After you created final_cicalino and final_imola
# --- Create final_cicalino ---
combineed_Cicalino1_features = add_features(combineed_Cicalino1)
combineed_Cicalino2_features = add_features(combineed_Cicalino2)

combineed_Cicalino1_features['location'] = 'Cicalino 1'
combineed_Cicalino2_features['location'] = 'Cicalino 2'

final_cicalino = pd.concat([combineed_Cicalino1_features, combineed_Cicalino2_features], ignore_index=True)
final_cicalino['location'] = 'Cicalino'  # unify location if you want one name

# --- Create final_imola ---
capture_i1 = pd.read_excel("/content/drive/MyDrive/Locations/grafico-delle-catture (Imola 1).xlsx")
weather_i1 = pd.read_excel("/content/drive/MyDrive/Locations/dati-meteo-storici (Imola 1).xlsx", header=[0, 1])
capture_i2 = pd.read_excel("/content/drive/MyDrive/Locations/grafico-delle-catture (Imola 2).xlsx")
weather_i2 = pd.read_excel("/content/drive/MyDrive/Locations/dati-meteo-storici (Imola 2).xlsx", header=[0, 1])
capture_i3 = pd.read_excel("/content/drive/MyDrive/Locations/grafico-delle-catture (Imola 3).xlsx")
weather_i3 = pd.read_excel("/content/drive/MyDrive/Locations/dati-meteo-storici (Imola 3).xlsx", header=[0, 1])

df_i1 = merge_weather_with_capture(clean_weather_data(weather_i1), clean_capture(capture_i1))
df_i2 = merge_weather_with_capture(clean_weather_data(weather_i2), clean_capture(capture_i2))
df_i3 = merge_weather_with_capture(clean_weather_data(weather_i3), clean_capture(capture_i3))

final_imola = pd.concat([df_i1, df_i2, df_i3], ignore_index=True)
final_imola = add_features(final_imola)
final_imola['location'] = 'Imola'




# Merge Cicalino and Imola
final_dataset = pd.concat([final_cicalino, final_imola], ignore_index=True)

# Sort by datetime if needed
final_dataset = final_dataset.sort_values('datetime').reset_index(drop=True)
import numpy as np

if current_page == "Final Dataset":
    st.header("📍 Final Dataset (Cicalino + Imola Combined)")



    st.subheader("EDA on Final Dataset")
    run_eda(final_dataset)
    plot_feature_correlation_heatmap(final_dataset, target_col="insect_count")

    # Optional: Display number of rows
    st.success(f"Total records: {len(final_dataset)}")


if current_page == "Regression Models":

    with st.expander("Random Forest Model on Final Dataset"):
        import numpy as np
        import pandas as pd
        import statsmodels.api as sm
        from statsmodels.discrete.count_model import ZeroInflatedPoisson
        from statsmodels.genmod.families import Poisson
        from statsmodels.api import OLS
        from sklearn.model_selection import train_test_split, cross_val_score
        from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
        from sklearn.metrics import mean_squared_error, r2_score

        st.subheader("Dataset Overview")
        st.info(f"Total rows available for modeling: {len(final_dataset)}")

        if len(final_dataset) < 100:
            st.warning("Dataset too small for reliable modeling. Consider adding more data.")
        else:
            # Prepare features
            final_dataset = add_features(final_dataset)

            # Create lag_1
            final_dataset = final_dataset.sort_values('datetime')
            final_dataset['lag_1'] = final_dataset['insect_count'].shift(1)

            features = ['rolling_mean','avg_humidity', 'avg_temp']
            target = 'insect_count'

            df_regression = final_dataset.dropna(subset=features + [target]).copy()

            def run_zero_aware_regressors(df, features, target='insect_count'):
                df = df.copy()
                df = df[features + [target]].dropna()
                X = df[features]
                y = df[target]

                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

                models = {
                    "Random Forest": RandomForestRegressor(max_depth=4, random_state=42),
                    # "HistGradientBoosting": HistGradientBoostingRegressor(max_depth=3, learning_rate=0.03, max_iter=300, random_state=42)
                }

                results = {}

                for name, model in models.items():
                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_test)

                    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
                    r2 = r2_score(y_test, y_pred)
                    cv_rmse = -cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=5).mean()

                    results[name] = {
                        "Model": name,
                        "Test RMSE": rmse_test,
                        "R² Test": r2,
                        "CV RMSE": cv_rmse
                    }

                return pd.DataFrame(results).T.sort_values("R² Test", ascending=False)

            def check_model_overfitting(model, X, y):
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
                model.fit(X_train, y_train)
                y_train_pred = model.predict(X_train)
                y_test_pred = model.predict(X_test)

                rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
                rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
                r2_train = r2_score(y_train, y_train_pred)
                r2_test = r2_score(y_test, y_test_pred)
                cv_rmse = -cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=5).mean()

                return {
                    "Train RMSE": rmse_train,
                    "Test RMSE": rmse_test,
                    "Train R²": r2_train,
                    "Test R²": r2_test,
                    "CV RMSE": cv_rmse
                }

            # def run_statsmodels_regressors(df, features, target='insect_count'):
            #     df = df.copy().dropna(subset=features + [target])
            #     X = df[features]
            #     y = df[target]
            #     X_const = sm.add_constant(X)

            #     results = {}

            #     poisson_model = sm.GLM(y, X_const, family=Poisson()).fit()
            #     results['Poisson'] = poisson_model

            #     # zip_model = ZeroInflatedPoisson(endog=y, exog=X_const, exog_infl=X_const, inflation='logit').fit(method='bfgs', maxiter=100)
            #     # results['Zero-Inflated Poisson'] = zip_model

            #     # ols_model = OLS(y, X_const).fit()
            #     # results['OLS'] = ols_model

            #     return results


            st.subheader(" Regression models ")
            results_df = run_zero_aware_regressors(df_regression, features)
            st.dataframe(results_df)

            # # Check overfitting on best model
            # st.subheader("Overfitting Check (HistGradientBoosting)")
            # model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.02, max_depth=2, early_stopping=True, validation_fraction=0.2, random_state=42)
            # metrics = check_model_overfitting(model, df_regression[features], df_regression[target])
            # st.json(metrics)

            # Run statsmodels regressors

    # with st.expander("Poisson Regression Summary"):
    #     statsmodels_results = run_statsmodels_regressors(df_regression, features)

    #     st.text(statsmodels_results['Poisson'].summary())



    import statsmodels.api as sm
    from statsmodels.discrete.count_model import ZeroInflatedPoisson
    from sklearn.ensemble import GradientBoostingRegressor
    from sklearn.linear_model import PoissonRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


    from sklearn.linear_model import PoissonRegressor, TweedieRegressor
    from sklearn.ensemble import HistGradientBoostingRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



    with st.expander(" Time Series Modeling on Final Dataset "):


        import itertools
        import statsmodels.api as sm
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        import numpy as np
        import matplotlib.pyplot as plt
        from sklearn.metrics import mean_squared_error, r2_score
        from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

        # Step 1: Prepare daily data
        daily_final = final_dataset.groupby('datetime').agg({
            'insect_count': 'sum',
            'avg_temp': 'mean',
            'avg_humidity': 'mean'
        }).sort_index()

        y = daily_final['insect_count']
        exog = daily_final[['avg_temp', 'avg_humidity']]

        st.subheader("Original Daily Insect Count")
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(y, marker='o')
        st.pyplot(fig)

        # Force first differencing manually
        st.warning("⚡ Forcing First Differencing (d=1)")
        y = y.diff().dropna()

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(y, marker='o')
        ax.set_title("After Forced First Differencing")
        st.pyplot(fig)

        st.success(" Final Differencing Order: d=1")

        # Plot ACF and PACF
        st.subheader("ACF and PACF After Differencing")
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        plot_acf(y, lags=20, ax=axes[0])
        plot_pacf(y, lags=20, ax=axes[1])
        st.pyplot(fig)

        # Step 2: SARIMAX Grid Search
        st.subheader("SARIMAX Hyperparameter Tuning (Small Search)")

        pdq = list(itertools.product(range(0, 3), [1], range(0, 3)))  # only d=1
        seasonal_pdq = [(0, 1, 1, 7), (1, 1, 1, 7)]  # mild seasonality

        best_score = float("inf")
        best_order = None
        best_seasonal_order = None
        best_model = None

        for order in pdq:
            for seasonal_order in seasonal_pdq:
                try:
                    model = SARIMAX(
                        y,
                        exog=exog.iloc[1:],  # Align after differencing
                        order=order,
                        seasonal_order=seasonal_order,
                        enforce_stationarity=False,
                        enforce_invertibility=False
                    )
                    results = model.fit(disp=False)

                    pred = results.predict(
                        start=y.index[int(len(y) * 0.8)],
                        end=y.index[-1],
                        exog=exog.iloc[int(len(y) * 0.8) + 1:]
                    )

                    rmse = np.sqrt(mean_squared_error(y.iloc[int(len(y) * 0.8):], pred))

                    if rmse < best_score:
                        best_score = rmse
                        best_order = order
                        best_seasonal_order = seasonal_order
                        best_model = results
                except Exception as e:
                    continue

        if best_model is not None:
            st.success(f" Best SARIMAX Order: {best_order} with Seasonal {best_seasonal_order}")

            # Step 3: Forecast
            split_idx = int(len(y) * 0.8)
            y_test = y.iloc[split_idx:]
            exog_test = exog.iloc[split_idx+1:]

            forecast = best_model.predict(
                start=y_test.index[0],
                end=y_test.index[-1],
                exog=exog_test
            )


            fig, ax = plt.subplots(figsize=(12, 6))
            ax.plot(y, label="Observed", marker='o')
            ax.plot(y_test.index, forecast, label="Forecast", color='red', marker='x')
            ax.legend()
            ax.set_title("Observed vs Forecasted (Final SARIMAX)")
            st.pyplot(fig)

            # Step 4: Evaluation
            st.subheader("Model Evaluation on Test Set")

            rmse = np.sqrt(mean_squared_error(y_test, forecast))
            r2 = r2_score(y_test, forecast)

            st.info(f"**RMSE:** {rmse:.3f}")
            st.info(f"**R² Score:** {r2:.3f}")

            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(y_test.index, y_test, label="Actual", marker='o')
            ax.plot(y_test.index, forecast, label="Predicted", linestyle='--', marker='x')
            ax.legend()
            st.pyplot(fig)

        else:
            st.error(" No stable SARIMAX model found after small search.")



if current_page == "Classification Models":

    st.markdown("### Classification Models on Final dataset")

    from imblearn.over_sampling import SMOTE
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression
    from xgboost import XGBClassifier
    import lightgbm as lgb
    from catboost import CatBoostClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier



      # 1. Prepare classification data
    features = ['avg_temp', 'avg_humidity', 'lag_2', 'rolling_mean']
    final_dataset['label'] = (final_dataset['insect_count'] > 0).astype(int)

    X = final_dataset[features]
    y = final_dataset['label']

    # 2. Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

    # 3. SMOTE (handle imbalance)
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

    # 4. Evaluation function
    def evaluate_classifier(name, y_true, y_pred):
        st.markdown(f"#### {name}")
        st.write(f"**Accuracy**: {accuracy_score(y_true, y_pred):.2f}")
        st.write(f"**Precision**: {precision_score(y_true, y_pred):.2f}")
        st.write(f"**Recall**: {recall_score(y_true, y_pred):.2f}")
        st.write(f"**F1 Score**: {f1_score(y_true, y_pred):.2f}")
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots()
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
        st.pyplot(fig)

    # 5. Train and test multiple classifiers
    with st.expander("Random Forest Classifier"):
        rf = RandomForestClassifier(random_state=42)
        rf.fit(X_train_bal, y_train_bal)
        y_pred_rf = rf.predict(X_test)
        evaluate_classifier("Random Forest", y_test, y_pred_rf)

    with st.expander("XGBoost Classifier"):
        xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
        xgb.fit(X_train_bal, y_train_bal)
        y_pred_xgb = xgb.predict(X_test)
        evaluate_classifier("XGBoost", y_test, y_pred_xgb)

    with st.expander("LightGBM Classifier"):
        lgbm = lgb.LGBMClassifier(random_state=42)
        lgbm.fit(X_train_bal, y_train_bal)
        y_pred_lgbm = lgbm.predict(X_test)
        evaluate_classifier("LightGBM", y_test, y_pred_lgbm)

    with st.expander(" Best Classification Model Summary"):
      st.markdown("### 🌟 Best Model: LightGBM")

      st.success(
          """
          - **Accuracy**: 0.83
          - **Precision**: 0.70
          - **Recall**: 0.70
          - **F1 Score**: 0.70


          """
      )


    with st.expander("🔧 Fine-tune LightGBM (Best Model)"):

        from sklearn.model_selection import RandomizedSearchCV
        import lightgbm as lgb

        param_grid = {
            'num_leaves': [20, 31, 40, 50],
            'max_depth': [-1, 5, 10, 15],
            'learning_rate': [0.01, 0.05, 0.1],
            'n_estimators': [100, 200, 500],
            'min_child_samples': [10, 20, 30],
            'subsample': [0.7, 0.8, 0.9],
            'colsample_bytree': [0.7, 0.8, 0.9]
        }

        lgbm_clf = lgb.LGBMClassifier(random_state=42)

        random_search = RandomizedSearchCV(
            lgbm_clf,
            param_distributions=param_grid,
            n_iter=30,
            scoring='f1',
            cv=3,
            verbose=1,
            random_state=42,
            n_jobs=-1
        )
        random_search.fit(X_train_bal, y_train_bal)

        best_lgbm = random_search.best_estimator_

        y_pred_best = best_lgbm.predict(X_test)

        st.success(f"✅ Best Hyperparameters found: {random_search.best_params_}")

        evaluate_classifier("Tuned LightGBM", y_test, y_pred_best)





Overwriting streamlit_app.py


In [10]:
from pyngrok import ngrok
ngrok.set_auth_token("2wLqFPF5EXdruJ5BgU6IfH33MNm_5e3Yvb1SSn9yJPswUfLw")


In [11]:
!ngrok config add-authtoken 2wLqFPF5EXdruJ5BgU6IfH33MNm_5e3Yvb1SSn9yJPswUfLw


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [12]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("Streamlit app URL:", public_url)

!streamlit run streamlit_app.py &> /dev/null &


Streamlit app URL: NgrokTunnel: "https://86d9-34-125-59-119.ngrok-free.app" -> "http://localhost:8501"
